# VALID relation mapping audit
Attach the exact UPT source and prompt v175 datasets. CPU only.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json

UPT_ROOT = Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2')
MAPPED_ROOT = Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')

def pinned(root, name, expected):
    matches = list(root.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Attach dataset or correct path: {root}/{name}')
    path = matches[0]
    with path.open('rb') as stream:
        if hashlib.file_digest(stream, 'sha256').hexdigest() != expected:
            raise ValueError(f'Unexpected SHA256: {path}')
    return path

source = pinned(UPT_ROOT, 'prompt_cache.json', '24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d')
links = pinned(MAPPED_ROOT, 'prompt_links.jsonl', 'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0')
upt = json.loads(source.read_text(encoding='utf-8'))
stats = defaultdict(Counter)
images = set()
lost_rows = []

def key(relation):
    return tuple(relation[k] for k in ('subject_instance', 'predicate', 'object_instance'))

with links.open(encoding='utf-8') as stream:
    for row in map(json.loads, stream):
        if row['group'] != 'val':
            continue
        cid = row['coco_id']
        if cid in images:
            raise ValueError(f'Duplicate VALID image {cid}')
        images.add(cid)
        entry = upt.get(row['filename'], {'objects': [], 'relations': []})
        objects = {o['instance_id']: o for o in entry['objects']}
        original = {key(r): r for r in entry['relations']}
        retained = {key(r) for r in row['relations']}
        if (len(original) != row['counts']['relations'] or not retained <= original.keys()
                or len(retained) != row['counts']['mapped_relations']):
            raise ValueError(f'Inconsistent mapping for {cid}')
        people = sum(o['label'] == 'person' for o in objects.values())
        crowd = 'multiple_people' if people > 1 else 'one_person'
        mapped_ids = {o['instance_id'] for o in row['objects']}
        for k, relation in original.items():
            box = objects[k[0]]['bbox']
            area = (box[2]-box[0])*(box[3]-box[1])
            size = 'small_under_1pct' if area < .01 else 'at_least_1pct'
            kept = k in retained
            for group in ('all', size, crowd, f'{size}/{crowd}'):
                stats[group]['relations'] += 1
                stats[group]['lost'] += int(not kept)
            if not kept:
                lost_rows.append(dict(coco_id=cid, filename=row['filename'], text=relation['text'],
                                      subject_bbox=box, object_bbox=objects[k[2]]['bbox'],
                                      subject_area=area, upt_people=people,
                                      subject_in_kept_objects=k[0] in mapped_ids,
                                      object_in_kept_objects=k[2] in mapped_ids,
                                      image_counts=row['counts']))

if len(images) != 5000 or stats['all'] != {'relations': 2526, 'lost': 207}:
    raise ValueError('VALID coverage differs from the reviewed v175 summary')
summary = {group: dict(counts, loss_percent=100*counts['lost']/counts['relations'])
           for group, counts in sorted(stats.items())}
result = dict(images=len(images), summary=summary, lost_relations=lost_rows)
output = Path('/kaggle/working/csa_relation_mapping_valid_audit.json')
output.write_text(json.dumps(result, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print(output)
